# AbstractTensor — Manifold, tangent bundle and frame bundle

This notebook explains the structures defined with @def_manifold macro and their hierarchy 

---
## 1. Loading package

In [1]:
using SymbolicTensors

[ Info: Precompiling SymbolicTensors [55c8b40c-cbfa-4141-803e-4831c9841971] (cache misses: include_dependency fsize change (2), wrong source (2), mismatched flags (14))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


---
## 2. Manifolds and def_manifold macro
---

### a. Manifold type and definition

#### Documentation access
- `@doc Manifold` — shows the docstring directly (julia macro)
- `?Manifold` — same, with search results prepended (Jupyter help mode)

In [2]:
@doc Manifold

```julia
Manifold
```

Struct representing a registered differentiable manifold. Instances are created by [`@def_manifold`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
M.name              # :M
M.dim               # 4
M.tangent_bundle    # :tangentM
M.cotangent_bundle  # :cotangentM
M.vbundles          # [:tangentM, :cotangentM]
```

### Fields

  * `name`             : manifold name, e.g. `:M`
  * `dim`              : dimension
  * `tangent_bundle`   : name of the tangent bundle, e.g. `:tangentM`
  * `cotangent_bundle` : name of the cotangent (dual) bundle, e.g. `:cotangentM`
  * `vbundles`         : names of vector bundles with manifold as base manifold


In [3]:
?@def_manifold

```julia
@def_manifold name dim coord_indices basis_indices [kwargs...]
```

Define a new manifold and automatically create its tangent and cotangent bundles, coordinate frames, and moving frame bundles.

Both index lists are **required**. Each list should have at least 4 symbols (a warning is issued if fewer).

Bind in the caller's scope:

  * `name`            → a [`Manifold`](@ref) instance
  * `tangent<name>`   → a [`VBundle`](@ref) instance (`isdual = false`)
  * `cotangent<name>` → a [`VBundle`](@ref) instance (`isdual = true`)
  * `frame<name>`, `coframe<name>`, moving basis symbols (default `e`, `θ`)

Coordinate indices (first list) → [`CoordinateIndex`](@ref):

```julia
a1          # CoordinateIndex(:a1, :tangentM)   — contravariant
-a1         # CoordinateIndex(:a1, :cotangentM) — covariant
```

Basis indices (second list) → [`BasisIndex`](@ref):

```julia
A1          # BasisIndex(:A1, :tangentM)   — contravariant
-A1         # BasisIndex(:A1, :cotangentM) — covariant
```

#### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
@def_manifold M d [b1, b2, b3, b4] [B1, B2, B3, B4]   # parametric dimension
```


In [4]:
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]

Defined manifold M of dimension 4
Defined vector bundle tangentM with coordinate basis ∂ and its dual cotangentM with coordinate basis dx
Defined frame bundle frameM (moving frame e) and coframe bundle coframeM (moving coframe θ) over M


In [5]:
_MANIFOLDS

Dict{Symbol, Manifold} with 1 entry:
  :M => Manifold(M, dim=4, TBundle=tangentM, CBundle=cotangentM)

### b. (co)Tangent bundle and (co)Frame bundle 

In [6]:
@doc VBundle

```julia
VBundle
```

Struct representing a registered vector bundle. Instances are created by [`@def_manifold`](@ref) for the tangent and cotangent bundles, and bound to variables in the caller's scope (e.g. `tangentM`, `cotangentM`).

Provides dot access to all metadata:

```julia
tangentM.name      # :tangentM
tangentM.manifold  # :M
tangentM.dim       # 4
tangentM.isdual    # false
tangentM.dual      # :cotangentM
tangentM.coordinate_indices  # [CoordinateIndex(:a1, :tangentM), ...]
tangentM.basis_indices       # [BasisIndex(:A1, :tangentM), ...]
tangentM.bases     # [Basis(:∂, :tangentM, :coordinate), Basis(:e, :tangentM, :moving)]
```

### Fields

  * `name`     : bundle name, e.g. `:tangentM`
  * `manifold` : base manifold name, e.g. `:M`
  * `dim`      : fibre dimension
  * `isdual`   : false = contravariant (upper) slots, true = covariant (lower) slots.              Authoritative for index variance via [`is_up`](@ref) / [`is_down`](@ref).
  * `dual`     : name of the paired dual bundle, e.g. `:cotangentM` or `:dualE`
  * `coordinate_indices` : [`CoordinateIndex`](@ref) for the coordinate chart (∂ / `dx`);              nonempty on tangent/cotangent bundles from [`@def_manifold`](@ref)
  * `basis_indices` : [`BasisIndex`](@ref) for fibre / moving bases (`e` / `θ`);              populated on tangent/cotangent by [`@def_manifold`](@ref) and on custom              bundles by [`@def_vbundle`](@ref)


In [7]:
tangentM

VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4, bases=[Basis(∂, tangentM, coord), Basis(e, tangentM, moving)])

In [8]:
(tangentM.bases,tangentM.coordinate_indices,tangentM.basis_indices)

(Basis[Basis(∂, tangentM, coord), Basis(e, tangentM, moving)], CoordinateIndex[+a1 ∈ tangentM (coord), +a2 ∈ tangentM (coord), +a3 ∈ tangentM (coord), +a4 ∈ tangentM (coord)], BasisIndex[+A1 ∈ tangentM (basis), +A2 ∈ tangentM (basis), +A3 ∈ tangentM (basis), +A4 ∈ tangentM (basis)])

In [9]:
(cotangentM,tangentM)

(VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=4, bases=[Basis(dx, cotangentM, coord), Basis(θ, cotangentM, moving)]), VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4, bases=[Basis(∂, tangentM, coord), Basis(e, tangentM, moving)]))

In [10]:
(cotangentM.bases,cotangentM.coordinate_indices,cotangentM.basis_indices)

(Basis[Basis(dx, cotangentM, coord), Basis(θ, cotangentM, moving)], CoordinateIndex[-a1 ∈ cotangentM (coord), -a2 ∈ cotangentM (coord), -a3 ∈ cotangentM (coord), -a4 ∈ cotangentM (coord)], BasisIndex[-A1 ∈ cotangentM (basis), -A2 ∈ cotangentM (basis), -A3 ∈ cotangentM (basis), -A4 ∈ cotangentM (basis)])

In [11]:
@doc Basis

```julia
Basis
```

A named frame for a vector bundle, of a given type (`:coordinate` or `:moving`). Instances are created by [`@def_manifold`](@ref) (coordinate and moving) or standalone [`@def_frame_bundle`](@ref) (moving only for custom bundles).

### Fields

  * `name`    : display name, e.g. `:dx`, `:∂`, `:e`, `:θ`
  * `vbundle` : the bundle this frame is for, e.g. `:cotangentM`
  * `type`    : `:coordinate` (natural frame) or `:moving` (user-defined frame)

Indexing a `Basis` with an [`AbstractIndex`](@ref) from the **dual** bundle produces a [`BasisElement`](@ref):

```julia
dx[a1]    # a1 ∈ tangentM  → BasisElement of cotangentM coordinate frame
e[-a1]    # -a1 ∈ cotangentM → BasisElement of tangentM moving frame
```


In [12]:
cotangentM.bases

2-element Vector{Basis}:
 Basis(dx, cotangentM, coord)
 Basis(θ, cotangentM, moving)

In [13]:
(dx,θ)

(Basis(dx, cotangentM, coord), Basis(θ, cotangentM, moving))

In [14]:
tangentM.bases

2-element Vector{Basis}:
 Basis(∂, tangentM, coord)
 Basis(e, tangentM, moving)

In [15]:
(∂,e)

(Basis(∂, tangentM, coord), Basis(e, tangentM, moving))

In [16]:
@doc BasisElement

```julia
BasisElement
```

A single element of a frame (coordinate or moving), constructed by `getindex` on a [`Basis`](@ref).

### Fields

  * `basis` : the [`Basis`](@ref) this element belongs to
  * `index` : the [`AbstractIndex`](@ref) labeling this element;           its vbundle is the **dual** of `basis.vbundle`

    dx[a1]   → BasisElement(Basis(:dx, :cotangentM, :coordinate), CoordinateIndex(:a1, :tangentM))   θ[a1]    → BasisElement(Basis(:θ,  :cotangentM, :moving),     CoordinateIndex(:a1, :tangentM))   ∂[-a1]   → BasisElement(Basis(:∂,  :tangentM,   :coordinate), CoordinateIndex(:a1, :cotangentM))   e[-a1]   → BasisElement(Basis(:e,  :tangentM,   :moving),     CoordinateIndex(:a1, :cotangentM))


In [17]:
(dx[a1],∂[-a1])

(dx[a1], ∂[-a1])

In [18]:
(θ[A1],e[-A1])

(θ[A1], e[-A1])

#### Error handling 

In [19]:
dx[-a1]

LoadError: BasisElement: index vbundle :cotangentM is not the dual of basis vbundle :cotangentM. Expected index from :tangentM. Use dx[...] with an index from the dual bundle.

In [39]:
dx[A1]

dx[A1]

#### Change of basis coordinate <-> moving

### c. Frame bundle 

In [21]:
frameM

VBundle,tangentM
Dual,coframeM
Moving basis,e


In [22]:
coframeM

VBundle,cotangentM
Dual,frameM
Moving basis,θ


---
## 3. Tensors
---

### a. Type and definition

In [23]:
@doc Tensor

```julia
Tensor
```

A registered abstract tensor. Instances are created by [`@def_tensor`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
T.manifold       # :M  (Symbol key into _MANIFOLDS)
T.slots          # [:cotangentM, :cotangentM]  — vbundle per slot
T.symmetries     # [SlotSymmetry(n=2, order=2)]
T.is_traceless   # false
T.known_traces   # Any[]  (populated later)
T.print_as       # :T
T.metric         # :g or nothing
T.rank           # 2      (derived — length of slots)
```

### Fields

  * `manifold`      : name of the base manifold, key into `_MANIFOLDS`
  * `slots`         : vbundle symbol per slot, encoding variance.                   `:cotangentM` → covariant, `:tangentM` → contravariant.
  * `symmetries`    : `Vector{`[`SlotSymmetry`](@ref)`}` — list of permutation                   groups acting on slot positions `1:rank`.
  * `is_traceless`  : if `true`, any self-contraction of this tensor gives zero                   (e.g. the Weyl tensor).
  * `known_traces`  : user-declared trace values, e.g. `g[a,-a] = dim`.                   Format TBD — stored as `Any[]` until contraction is                   implemented.
  * `print_as`      : display name. Controls how the tensor appears in `show`                   and (later) LaTeX output.
  * `metric`        : name of the metric tensor associated with this tensor,                   a key into `_METRICS`, or `nothing` if no metric                   has been assigned. Required for raising/lowering indices.


In [24]:
@doc @def_tensor

```julia
@def_tensor name [vbundle1, vbundle2, ...] [symmetries=...] [traceless=...] [print_as=:...] [metric=...]
```

Define a new abstract tensor and bind it to `name` in the caller's scope.

### Slot syntax

Specify slots directly as VBundle symbols. The manifold is automatically deduced from the first VBundle's `manifold` field.

  * `:tangentM` → contravariant (upper) slot
  * `:cotangentM` → covariant (lower) slot
  * Any user-defined VBundle from `@def_vbundle`

All VBundle symbols must belong to the same manifold.

### Keyword arguments (all optional)

  * `symmetries`  : a [`SlotSymmetry`](@ref) or `Vector{SlotSymmetry}` describing                 the slot permutation symmetry group(s). Defaults to                 `[no_symmetry(rank)]`.
  * `traceless`   : `true` if any self-contraction of this tensor is zero                 (e.g. Weyl tensor). Defaults to `false`.
  * `print_as`    : display name. Defaults to `name`. Example: `print_as=:R`.
  * `metric`      : name of the metric tensor to associate with this tensor.                 Omitting this keyword triggers automatic resolution:                 - one metric on manifold → assigned silently                 - multiple metrics → `@warn`, first defined is assigned                 - no metric → `@warn`, `nothing` assigned (no raising/lowering)

Binds `name` to a [`Tensor`](@ref) instance in the caller's scope and registers it in [`_TENSORS`](@ref).

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]
@def_metric g M

@def_tensor T [cotangentM, cotangentM]                 # (0,2) tensor, metric auto-resolved
@def_tensor F [cotangentM, cotangentM] symmetries=[antisymmetric(2)]
@def_tensor R [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()]
@def_tensor W [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()] traceless=true print_as=:Weyl
@def_tensor mixed_T [tangentM, cotangentM]            # (1,1) mixed tensor
```


In [25]:
cotangentM

VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=4, bases=[Basis(dx, cotangentM, coord), Basis(θ, cotangentM, moving)])

In [26]:
_VBUNDLES

Dict{Symbol, VBundle} with 2 entries:
  :cotangentM => VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=…
  :tangentM   => VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4,…

In [27]:
@def_tensor T [cotangentM, cotangentM]

┌ Warning: No metric is defined on manifold M. Tensor T has no metric assigned; indices cannot be raised or lowered.
└ @ Main ~/JuliaTensor/SymbolicTensors/src/tensors.jl:325


Defined tensor T on manifold M with 2 slot(s), metric=nothing


In [35]:
T

Manifold,M
Rank,2
Slots,↓cotangentM ↓cotangentM
Symmetries,NoSymmetry(n=2)
Traceless,false
Metric,none


In [28]:
(T.is_traceless,T.known_traces,T.manifold,T.metric,T.print_as,T.rank,T.slots,T.symmetries)

(false, Any[], :M, nothing, :T, 2, [:cotangentM, :cotangentM], SlotSymmetry[NoSymmetry(n=2)])

In [29]:
T.symmetries

1-element Vector{SlotSymmetry}:
 NoSymmetry(n=2)

In [30]:
T.slots

2-element Vector{Symbol}:
 :cotangentM
 :cotangentM

In [31]:
T[-a1,-a2]

T[-a1, -a2]

In [32]:
(T[-a1,-a2],T[a1,-a2],T[-a1,a2],T[a1,a2],T[-A1,-A2],T[A1,-a1],T[A1,a1])

(T[-a1, -a2], T[a1, -a2], T[-a1, a2], T[a1, a2], T[-A1, -A2], T[A1, -a1], T[A1, a1])

##### Error handling 

In [33]:
T[-a1,-a1]

T[-a1, -a1]

In [34]:
T[a1,a1]

T[a1, a1]

In [36]:
@doc basis_expansion

```julia
basis_expansion(T::Tensor, b::Basis) -> BasisExpansion
```

Expand the abstract tensor `T` in the frame described by the [`Basis`](@ref) object `b`, using the canonical slot index assignment declared at `@def_tensor` time.

This is the most type-safe entry point: `b` is a concrete registered `Basis` struct bound in the user's scope by `@def_manifold` or `@def_frame_bundle`.

For each slot `i`:

  * The component index is the `i`-th registered coordinate symbol placed in `T.slots[i]` (i.e. `CoordinateIndex(:a_i, slot_vb)`).
  * The basis element uses `b` and labels it with the *dual* index `CoordinateIndex(:a_i, dual_of_slot_vb)`.

This means `b.vbundle` must match every slot's vbundle in `T.slots`. If `T` has mixed slots (e.g. `[tangentM, cotangentM]`), pass the frame for each slot separately or use the `frame::Symbol` overload which handles per-slot lookup automatically.

# Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
@def_metric   g[-a1, -a2] M
@def_tensor   T[-a1, -a2] M

basis_expansion(T, dx)   # T[-a1,-a2] dx[a1] ⊗ dx[a2]
basis_expansion(T, θ)    # T[-a1,-a2] θ[a1]  ⊗ θ[a2]
```


In [37]:
(basis_expansion(T),basis_expansion(T,dx))

LoadError: MethodError: no method matching basis_expansion(::Tensor)
The function `basis_expansion` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  basis_expansion(::Tensor, [91m::Basis[39m)
[0m[90m   @[39m [36mAbstractTensors[39m [90m~/JuliaTensor/SymbolicTensors/src/[39m[90m[4mframes.jl:549[24m[39m


In [38]:
T(∂[-a1],∂[-a2])

LoadError: MethodError: objects of type Tensor are not callable
The object of type `Tensor` exists, but no method is defined for this combination of argument types when trying to treat it as a callable object.